# Group 2 Challenge: CDO Flood and Weather Dispatch

Your team is building an automated weather and flood alert system for local barangays.
Fix the broken starter code across all three tasks so every assertion passes.

**Scoring:** Task 1 (`30 pts`) + Task 2 (`35 pts`) + Task 3 (`35 pts`) = **`100 pts`**.

In [ ]:
SCORES = {"task_1": 0, "task_2": 0, "task_3": 0}
print("Score tracker initialized. Complete Tasks 1 to 3 below!")

---
## Task 1: Encapsulation `[30 pts]` — Protect `WeatherStation` State

Fix the `WeatherStation` class below to enforce these rules:
1. `__init__(self, station_id: str, threshold: float)`:
   - Raise `ValueError("threshold must be between 0 and 500")` if not `0 <= threshold <= 500`.
   - Store `self._station_id`, `self._threshold`, and `self._readings = []`.
2. Read-only properties:
   - `station_id` -> returns `self._station_id`
   - `readings` -> returns `tuple(self._readings)` (prevents callers from mutating internal list!)
3. Validated `threshold` property:
   - `@property` `threshold` returns `self._threshold`.
   - `@threshold.setter` raises `ValueError("threshold must be between 0 and 500")` if not `0 <= value <= 500`; otherwise updates `self._threshold`.
4. Method `record(self, value: float)`:
   - Raise `ValueError("reading must be between 0 and 500")` if not `0 <= value <= 500`.
   - Append `float(value)` to `self._readings`.

In [ ]:
class WeatherStation:
    def __init__(self, station_id: str, threshold: float):
        self.station_id = station_id
        self.threshold = threshold
        self.readings = []

    def record(self, value: float):
        self.readings.append(float(value))

    def risk_level(self) -> str:
        return "NORMAL"

    def report(self) -> str:
        return f"[{self._station_id}] Readings: {len(self._readings)}"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 1 (30 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    bad_init = False
    try:
        WeatherStation("ST-0", -5)
    except ValueError:
        bad_init = True
    assert bad_init, "WeatherStation('ST-0', -5) must raise ValueError."

    ws = WeatherStation("CDO-01", 80.0)
    assert hasattr(ws, "_station_id") and hasattr(ws, "_threshold") and hasattr(ws, "_readings"), "Use protected attributes (_station_id, _threshold, _readings)."
    assert ws.station_id == "CDO-01" and ws.threshold == 80.0, "Properties station_id and threshold must return initial values."

    ro = False
    try:
        ws.station_id = "NEW"
    except AttributeError:
        ro = True
    assert ro, "Property 'station_id' must be read-only."

    assert isinstance(ws.readings, tuple), "Property 'readings' must return a tuple."

    bad_set = False
    try:
        ws.threshold = 600
    except ValueError:
        bad_set = True
    assert bad_set, "Setting threshold > 500 must raise ValueError."

    bad_rec = False
    try:
        ws.record(-1)
    except ValueError:
        bad_rec = True
    assert bad_rec, "record(-1) must raise ValueError."

    ws.record(45)
    ws.record(90)
    assert ws.readings == (45.0, 90.0), f"Expected readings (45.0, 90.0), got {ws.readings}."

    SCORES["task_1"] = 30
    print("[PASS] Task 1 (30 / 30 pts): WeatherStation encapsulation verified.")
except Exception as e:
    SCORES["task_1"] = 0
    print(f"[FAIL] Task 1 (0 / 30 pts): {e}")

---
## Task 2: Inheritance `[35 pts]` — `RainGauge` and `RiverSensor`

Fix `RainGauge` and `RiverSensor` so they inherit from `WeatherStation`:
1. `RainGauge(WeatherStation)`:
   - `__init__(self, station_id: str, threshold: float, district: str)`: call `super().__init__(station_id, threshold)` and store `self.district = district`.
   - Override `risk_level(self) -> str`: return `"NO DATA"` if `self._readings` is empty; return `"CRITICAL RAIN"` if `max(self._readings) >= self._threshold`; else `"NORMAL"`.
   - Extend `report(self) -> str`: return `f"{super().report()} | Rain ({self.district}) -> {self.risk_level()}"`.
2. `RiverSensor(WeatherStation)`:
   - `__init__(self, station_id: str, threshold: float, river_name: str)`: call `super().__init__(station_id, threshold)` and store `self.river_name = river_name`.
   - Override `risk_level(self) -> str`: return `"NO DATA"` if `self._readings` is empty; return `"FLOOD ALERT"` if `self._readings[-1] >= self._threshold`; else `"SAFE"`.
   - Extend `report(self) -> str`: return `f"{super().report()} | River ({self.river_name}) -> {self.risk_level()}"`.

In [ ]:
class RainGauge(WeatherStation):
    def __init__(self, station_id: str, threshold: float, district: str):
        self.district = district

    def risk_level(self) -> str:
        return "NORMAL"

    def report(self) -> str:
        return f"Rain {self.district}"


class RiverSensor(WeatherStation):
    def __init__(self, station_id: str, threshold: float, river_name: str):
        self.river_name = river_name

    def risk_level(self) -> str:
        return "SAFE"

    def report(self) -> str:
        return f"River {self.river_name}"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 2 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert issubclass(RainGauge, WeatherStation) and issubclass(RiverSensor, WeatherStation), "Both subclasses must inherit from WeatherStation."

    rg = RainGauge("RG-1", 50.0, "Carmen")
    rs = RiverSensor("RV-1", 120.0, "Cagayan River")

    assert rg.station_id == "RG-1" and rg.risk_level() == "NO DATA", "RainGauge must call super().__init__() and return NO DATA when empty."
    assert rs.station_id == "RV-1" and rs.risk_level() == "NO DATA", "RiverSensor must call super().__init__() and return NO DATA when empty."

    rg.record(60.0)
    rg.record(20.0)
    assert rg.risk_level() == "CRITICAL RAIN", "RainGauge should trigger CRITICAL RAIN when max reading >= threshold."
    assert rg.report() == "[RG-1] Readings: 2 | Rain (Carmen) -> CRITICAL RAIN", f"Unexpected RainGauge.report(): {rg.report()}"

    rs.record(130.0)
    rs.record(95.0)
    assert rs.risk_level() == "SAFE", "RiverSensor checks the LATEST reading (_readings[-1]), so 95.0 < 120.0 is SAFE."
    assert rs.report() == "[RV-1] Readings: 2 | River (Cagayan River) -> SAFE", f"Unexpected RiverSensor.report(): {rs.report()}"

    SCORES["task_2"] = 35
    print("[PASS] Task 2 (35 / 35 pts): Inheritance and super() verified.")
except Exception as e:
    SCORES["task_2"] = 0
    print(f"[FAIL] Task 2 (0 / 35 pts): {e}")

---
## Task 3: Polymorphism and Duck Typing `[35 pts]` — `CitizenSMSReport` & `collect_alerts`

Fix the code below to achieve clean polymorphism:
1. `CitizenSMSReport` must **NOT** inherit from `WeatherStation` (standalone duck-typed class).
   - Rename/implement `risk_level(self) -> str` returning `f"SMS ({self.barangay}): {self.severity}"`.
2. Refactor `collect_alerts(monitors: list) -> list`:
   - Remove **all** `isinstance(...)` checks!
   - Loop through `monitors` and return a list of `m.risk_level()` strings.

In [ ]:
class CitizenSMSReport:
    def __init__(self, barangay: str, severity: str):
        self.barangay = barangay
        self.severity = severity

    def get_sms_text(self) -> str:
        return f"SMS ({self.barangay}): {self.severity}"


def collect_alerts(monitors: list) -> list:
    alerts = []
    for m in monitors:
        if isinstance(m, RainGauge):
            alerts.append(m.risk_level())
        elif isinstance(m, RiverSensor):
            alerts.append(m.risk_level())
    return alerts

In [ ]:
# RUN THIS CELL TO VERIFY TASK 3 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert not issubclass(CitizenSMSReport, WeatherStation), "CitizenSMSReport must NOT inherit from WeatherStation."
    assert hasattr(CitizenSMSReport, "risk_level"), "CitizenSMSReport must implement risk_level()."

    assert "isinstance" not in collect_alerts.__code__.co_names, "Remove all isinstance() calls from collect_alerts()."

    r = RainGauge("R1", 40.0, "Lapasan")
    r.record(50.0)
    rv = RiverSensor("V1", 100.0, "Iponan")
    rv.record(110.0)
    sms = CitizenSMSReport("Macasandig", "RISING WATER")

    out = collect_alerts([r, rv, sms])
    expected = ["CRITICAL RAIN", "FLOOD ALERT", "SMS (Macasandig): RISING WATER"]
    assert out == expected, f"Expected {expected}, got {out}."

    SCORES["task_3"] = 35
    print("[PASS] Task 3 (35 / 35 pts): Polymorphism and duck typing verified.")
except Exception as e:
    SCORES["task_3"] = 0
    print(f"[FAIL] Task 3 (0 / 35 pts): {e}")

---
## Final Scoreboard

In [ ]:
# RUN THIS CELL TO VIEW YOUR FINAL COMPETITION SCORE
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
total = SCORES["task_1"] + SCORES["task_2"] + SCORES["task_3"]
print("=" * 42)
print("DS212 GROUP COMPETITION SCOREBOARD")
print("=" * 42)
print(f"Task 1 (Encapsulation): {SCORES['task_1']:>2} / 30 pts")
print(f"Task 2 (Inheritance):   {SCORES['task_2']:>2} / 35 pts")
print(f"Task 3 (Polymorphism):  {SCORES['task_3']:>2} / 35 pts")
print("-" * 42)
print(f"TOTAL SCORE:           {total:>3} / 100 pts")
if total == 100:
    print("STATUS: ALL CHALLENGES COMPLETED [100%]")
else:
    print("STATUS: IN PROGRESS — KEEP DEBUGGING!")
print("=" * 42)